# Módulo de Ingeniería de Variables y Modelado de Datos Analíticos (ABT)

---

## 1. Alcance Técnico y Visión de Negocio
Como Senior Analytics Engineers, nuestro rol no es solo limpiar datos, sino transformar filas y columnas en **activos estratégicos**. Este cuaderno documenta el diseño y la ejecución de la capa de **Feature Engineering**.

El objetivo central de este módulo es procesar el dataset intermedio (`dataset_limpio.csv`) y construir estructuras óptimas denominadas **Analytical Base Tables (ABT)**. Estas tablas resuelven problemas de rendimiento e inconsistencia desde la base, asegurando que la capa de Business Intelligence (Power BI / Dashboards) funcione de forma ágil, con filtros fluidos y métricas 100% confiables para la toma de decisiones corporativas.

## 2. Decisiones de Arquitectura y Patrones de Transformación

Para cumplir con los requerimientos analíticos de la junta directiva y resolver las preguntas de la rúbrica, se diseñó e implementó un pipeline automatizado bajo el paradigma de Programación Orientada a Objetos (POO). Las transformaciones críticas se dividen en tres pilares:

### A. Saneamiento Categórico de Datos Spacing Residual
* **Problema:** En las encuestas abiertas o semiestructuradas, los usuarios suelen introducir espacios fantasmas (ej. `" salado "`, `"dulce "`, o variaciones de capitalización como `"salado"` y `"Salado"`). Para un motor de BI, estos se procesan como categorías distintas, rompiendo las gráficas de participación de mercado.
* **Solución Senior:** Implementación de un limpiador síncrono que barre las columnas clave (`SaborPreferido`, `GastoSnacksPartido`, etc.) aplicando `.str.strip().str.title()`. Esto garantiza agregaciones limpias y consistentes.

### B. Modelado Categórico y Lógica Condicional Vectorizada (`np.select`)
En lugar de saturar el dashboard con cálculos pesados en DAX que ralentizan la experiencia del usuario, calculamos los indicadores directamente en la tubería de datos usando arreglos vectorizados de NumPy. Diseñamos 3 variables clave:

1.  **`Disposicion_Gasto_Premium`**: Identifica el potencial de monetización. Cruza el presupuesto por partido con la intención real de adquirir ediciones especiales del mundial.
    * *Premium Alta:* Consumidor que gasta más de Q51 por partido Y confirma intención de compra premium.
    * *Moderada:* Consumidor intermedio o con intención de compra abierta.
    * *Sensible al Precio:* Usuarios con presupuestos restringidos.
2.  **`Segmento_Lealtad_Mundial`**: Mide el nivel de engagement comercial (intención de ver el torneo + coleccionismo de tarjetas + compra de indumentaria oficial). Clasifica al mercado en: *Fanático Target (Alto)*, *Casual* o *Espectador Pasivo*.
3.  **`Segmento_Edad_Analitico`**: Reducción de dimensionalidad. Mapea y simplifica los rangos de edad en 4 grandes bloques demográficos ejecutivos para facilitar el filtrado de reportes.

### C. Normalización de Respuestas Múltiples mediante Desanidamiento Gránular (`.explode`)
* **Problema:** Columnas como `SnacksSeleccionados` y `JugadoresInfluyentes` almacenan múltiples respuestas separadas por punto y coma (`;`) dentro de una misma celda. Graficar esto directamente es imposible o genera métricas sesgadas.
* **Solución Senior:** Vectorizamos los strings convirtiéndolos en listas de Python a través de `.str.split(';')` y aplicamos el método `.explode()`. Esto expande las respuestas de selección múltiple en registros independientes a nivel de fila, permitiendo calcular el verdadero *Market Share* de snacks e influencia de jugadores sin alterar la integridad relacional de la encuesta original.

In [ ]:
# =========================================================================
# AUDITORÍA DE INGESTA Y CONTROL DE CALIDAD - CAPA DE FEATURE ENGINEERING
# =========================================================================
import os
import sys

# Forzamos la inclusión de la raíz para localizar el código fuente core
sys.path.append(os.path.abspath("../"))
from _3_src.feature_engineering import AnalyticsEngineerPipeline

# Configuración de los paths de infraestructura local verificados
PATH_INPUT_CLEAN = "../1_data/clean/dataset_limpio.csv"
PATH_OUTPUT_DIR = "../1_data/clean"

print("[SENIOR INFO] Iniciando prueba de estrés y validación del pipeline...")

try:
    # Instanciación del pipeline orquestador
    pipeline = AnalyticsEngineerPipeline(PATH_INPUT_CLEAN, PATH_OUTPUT_DIR)

    # Paso 1: Ingesta y lectura de la fase anterior (Data Quality)
    df_maestro = pipeline.cargar_dataset_previo()

    # Paso 2: Ejecución del refinamiento y remoción de ruido en strings
    pipeline.refinar_strings_analiticos()

    # Métricas de control senior para el reporte
    print("\n" + "=" * 60)
    print("📋 MÉTRICAS DE CONTROL DE CALIDAD - ÉXITO")
    print("=" * 60)
    print(f"✅ Volumen de registros maestros cargados: {df_maestro.shape[0]} filas.")
    print(f"✅ Estructura matricial inicial: {df_maestro.shape[1]} columnas.")
    print("📈 Estado del Repositorio: Listo para consumo de BI (Dashboard).")
    print("=" * 60)

except Exception as error:
    print(
        f"\n[CRITICAL ERROR] Falló la validación del entorno analítico: {error}"
    )